<a href="https://colab.research.google.com/github/Himasree24/GENAI/blob/main/Skill/Week2_LLM_Client.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!mkdir -p week-2
%cd week-2
!pwd


/content/week-2
/content/week-2


In [ ]:

!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh


Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:

import subprocess
import time
import requests

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(30):
    try:
        r = requests.get("http://localhost:11434")
        if r.ok:
            print("Server says:", r.text)
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("Server did not start. Re-run this cell.")


Server says: Ollama is running


In [ ]:

!ollama pull llama3.2



In [ ]:

!ollama list


NAME               ID              SIZE      MODIFIED       
llama3.2:latest    a80c4f17acd5    2.0 GB    14 seconds ago    


In [ ]:

!curl -s http://localhost:11434/api/generate -d '{"model": "llama3.2", "prompt": "Say hello in one short sentence.", "stream": false}'


In [ ]:

import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2",
        "prompt": "Say hello in one short sentence.",
        "stream": False
    },
    timeout=180
)

print("Status:", response.status_code)
print("Response:", response.json()["response"])


ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7a40a9fd65d0>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [ ]:

import subprocess
import time
import requests

# Start Ollama server again
ollama_log = open("/tmp/ollama.log", "w")

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=ollama_log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

# Wait for the server
for i in range(30):
    try:
        response = requests.get(
            "http://localhost:11434",
            timeout=2
        )

        if response.ok:
            print("SUCCESS: Ollama is running!")
            break

    except requests.exceptions.ConnectionError:
        time.sleep(1)

else:
    print("Ollama failed to start.")
    print(open("/tmp/ollama.log").read())


ReadTimeout: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=2)

In [ ]:

import subprocess

print("Checking Ollama processes...\n")

subprocess.run(["pgrep", "-af", "ollama"])

print("\nChecking Ollama logs...\n")

subprocess.run(["tail", "-n", "30", "/tmp/ollama.log"])


Checking Ollama processes...


Checking Ollama logs...



CompletedProcess(args=['tail', '-n', '30', '/tmp/ollama.log'], returncode=0)

In [ ]:

import subprocess
import time
import requests

# Start Ollama server
with open("/tmp/ollama.log", "w") as log:
    ollama_proc = subprocess.Popen(
        ["ollama", "serve"],
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

print("Starting Ollama...")

# Wait for server response
for i in range(60):
    try:
        r = requests.get(
            "http://127.0.0.1:11434/",
            timeout=5
        )

        if r.status_code == 200:
            print("SUCCESS: Ollama is running!")
            break

    except requests.exceptions.RequestException:
        time.sleep(2)

else:
    print("Ollama did not respond.")
    print(open("/tmp/ollama.log").read())


Starting Ollama...
SUCCESS: Ollama is running!


In [ ]:

!ollama list


NAME               ID              SIZE      MODIFIED      
llama3.2:latest    a80c4f17acd5    2.0 GB    5 minutes ago    


In [ ]:

import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2",
        "prompt": "Say hello in one short sentence.",
        "stream": False
    },
    timeout=180
)

print("Status:", response.status_code)
print("Response:", response.json()["response"])


Status: 200
Response: Hello!


In [ ]:

from google.colab import userdata

key = userdata.get("GEMINI_API_KEY")

with open(".env", "w") as f:
    f.write(f'GEMINI_API_KEY="{key}"\n')

print(".env created. Key length:", len(key))


.env created. Key length: 53


In [ ]:
!ls -a

.  ..  .env


In [ ]:

%%writefile requirements.txt
python-dotenv
pytest
google-genai
requests


Writing requirements.txt


In [ ]:

!pip install -q -r requirements.txt


In [ ]:

from google import genai

print("google-genai imported OK")


google-genai imported OK


In [ ]:

%%writefile llm_client.py

from abc import ABC, abstractmethod
from google import genai
import requests


# Common interface: every provider must implement generate()
class LLMProvider(ABC):

    @abstractmethod
    def generate(self, prompt):
        pass


# Gemini cloud provider
class GeminiProvider(LLMProvider):

    def __init__(self, api_key, model="gemini-3.8-flash"):
        self.client = genai.Client(api_key=api_key)
        self.model = model

    def generate(self, prompt):
        response = self.client.models.generate_content(
            model=self.model,
            contents=prompt
        )
        return response.text


# Ollama local provider
class OllamaProvider(LLMProvider):

    def __init__(self, model="llama3.2"):
        self.model = model
        self.url = "http://localhost:11434/api/generate"

    def generate(self, prompt):
        response = requests.post(
            self.url,
            json={
                "model": self.model,
                "prompt": prompt,
                "stream": False
            },
            timeout=120
        )
        response.raise_for_status()
        return response.json()["response"]


# Common client: the rest of your program only talks to this class
class LLMClient:

    def __init__(self, provider):
        self.provider = provider

    def generate(self, prompt):
        return self.provider.generate(prompt)


Writing llm_client.py


In [ ]:

!ls


llm_client.py  requirements.txt


In [ ]:

%%writefile main.py

import os
from dotenv import load_dotenv

from llm_client import (
    GeminiProvider,
    OllamaProvider,
    LLMClient
)

load_dotenv(override=True)

provider_name = input(
    "Choose provider (gemini/ollama): "
).strip().lower()

if provider_name == "gemini":

    api_key = os.getenv("GEMINI_API_KEY")

    if not api_key:
        raise ValueError("Gemini API key not found in .env")

    provider = GeminiProvider(api_key)

elif provider_name == "ollama":

    provider = OllamaProvider("llama3.2")

else:
    raise ValueError("Invalid provider selected")

client = LLMClient(provider)

prompt = input("Enter your question: ")

answer = client.generate(prompt)

print("\nLLM Response:")
print(answer)


Writing main.py


In [ ]:

!ls -la


total 24
drwxr-xr-x 2 root root 4096 Nov 19 05:00 .
drwxr-xr-x 1 root root 4096 Nov 19 04:43 ..
-rw-r--r-- 1 root root   71 Nov 19 04:57 .env
-rw-r--r-- 1 root root 1377 Nov 19 04:59 llm_client.py
-rw-r--r-- 1 root root  708 Nov 19 05:00 main.py
-rw-r--r-- 1 root root   43 Nov 19 04:58 requirements.txt


In [ ]:

%%writefile test_client.py

import unittest
from llm_client import LLMClient


# Fake provider for testing
class FakeProvider:

    def generate(self, prompt):
        return "AI stands for Artificial Intelligence."


class TestLLMClient(unittest.TestCase):

    def test_generate(self):

        provider = FakeProvider()
        client = LLMClient(provider)

        result = client.generate("Explain AI")

        self.assertEqual(
            result,
            "AI stands for Artificial Intelligence."
        )


if __name__ == "__main__":
    unittest.main()


Writing test_client.py


In [ ]:

!python -m pytest test_client.py -v


============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week-2
plugins: typeguard-4.6.0, platformdirs-4.14.3, langsmith-0.14.17, anyio-4.16.0
collected 1 item                                                               

test_client.py::TestLLMClient::test_generate PASSED                      [100%]

============================== 1 passed in 1.75s ===============================


In [ ]:

!ls -la


total 36
drwxr-xr-x 4 root root 4096 Nov 19 05:00 .
drwxr-xr-x 1 root root 4096 Nov 19 04:43 ..
-rw-r--r-- 1 root root   71 Nov 19 04:57 .env
-rw-r--r-- 1 root root 1377 Nov 19 04:59 llm_client.py
-rw-r--r-- 1 root root  708 Nov 19 05:00 main.py
drwxr-xr-x 2 root root 4096 Nov 19 05:00 __pycache__
drwxr-xr-x 3 root root 4096 Nov 19 05:00 .pytest_cache
-rw-r--r-- 1 root root   43 Nov 19 04:58 requirements.txt
-rw-r--r-- 1 root root  540 Nov 19 05:00 test_client.py


In [ ]:

%run main.py


Choose provider (gemini/ollama): ollama
Enter your question: Explain AI in two sentences.

LLM Response:
Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making. By using algorithms and data, AI systems can analyze and process large amounts of information, often with the ability to improve their performance over time, enabling them to automate tasks, make predictions, and even create their own content.


In [ ]:

import time
from google.colab import userdata
from google import genai
from google.genai.errors import ServerError

key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=key)

for attempt in range(3):
    try:
        print(f"Attempt {attempt + 1}...")

        response = client.models.generate_content(
            model="gemini-3.8-flash",
            contents="Explain AI in two sentences."
        )

        print("\nGemini Response:")
        print(response.text)
        break

    except ServerError as e:
        print("Gemini server temporarily unavailable.")

        if attempt < 2:
            print("Retrying in 15 seconds...")
            time.sleep(15)
        else:
            print("Please try again later.")
            print(e)


Attempt 1...

Gemini Response:
Artificial intelligence (AI) refers to computer systems designed to mimic human cognitive abilities, such as learning, reasoning, and problem-solving. By analyzing vast amounts of data, these systems can recognize patterns and adapt over time to perform complex tasks that traditionally required human intellect.


In [ ]:
!python -m pytest test_client.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week-2
plugins: typeguard-4.6.0, platformdirs-4.14.3, langsmith-0.14.17, anyio-4.16.0
collected 1 item                                                               

test_client.py::TestLLMClient::test_generate PASSED                      [100%]

============================== 1 passed in 1.22s ===============================


In [ ]:
!ls -la

total 36
drwxr-xr-x 4 root root 4096 Nov 19 05:00 .
drwxr-xr-x 1 root root 4096 Nov 19 04:43 ..
-rw-r--r-- 1 root root   71 Nov 19 04:57 .env
-rw-r--r-- 1 root root 1377 Nov 19 04:59 llm_client.py
-rw-r--r-- 1 root root  708 Nov 19 05:00 main.py
drwxr-xr-x 2 root root 4096 Nov 19 05:00 __pycache__
drwxr-xr-x 3 root root 4096 Nov 19 05:00 .pytest_cache
-rw-r--r-- 1 root root   43 Nov 19 04:58 requirements.txt
-rw-r--r-- 1 root root  540 Nov 19 05:00 test_client.py


In [ ]:

!python -m pytest test_client.py -v


============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week-2
plugins: typeguard-4.6.0, platformdirs-4.14.3, langsmith-0.14.17, anyio-4.16.0
collected 1 item                                                               

test_client.py::TestLLMClient::test_generate PASSED                      [100%]

============================== 1 passed in 1.19s ===============================


In [ ]:
!ls -la

total 36
drwxr-xr-x 4 root root 4096 Nov 19 05:00 .
drwxr-xr-x 1 root root 4096 Nov 19 04:43 ..
-rw-r--r-- 1 root root   71 Nov 19 04:57 .env
-rw-r--r-- 1 root root 1377 Nov 19 04:59 llm_client.py
-rw-r--r-- 1 root root  708 Nov 19 05:00 main.py
drwxr-xr-x 2 root root 4096 Nov 19 05:00 __pycache__
drwxr-xr-x 3 root root 4096 Nov 19 05:00 .pytest_cache
-rw-r--r-- 1 root root   43 Nov 19 04:58 requirements.txt
-rw-r--r-- 1 root root  540 Nov 19 05:00 test_client.py
